# 🎬 Sistema de Recomendación de Películas
### Filtrado Colaborativo con KNN y SVD

Este notebook implementa un sistema de recomendación de películas usando dos enfoques:
- **KNN (K-Nearest Neighbors)**: filtrado colaborativo basado en usuarios similares.
- **SVD (Singular Value Decomposition)**: factorización de matrices con la librería Surprise.

**Dataset**: [MovieLens 100K](https://grouplens.org/datasets/movielens/100k/)

## 0. Instalación de dependencias
Ejecuta esta celda solo si estás en Google Colab.

In [ ]:
# Descomenta si usas Google Colab
# !pip install scikit-surprise matplotlib pandas numpy scikit-learn

## 1. Importaciones

In [ ]:
import sys
import os

# Añadir la raíz del proyecto al path para importar src/
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_preprocessing import load_data, clean_data, create_user_item_matrix
from src.recommender_knn import train_knn_model, get_knn_recommendations
from src.recommender_svd import train_svd_model, get_svd_recommendations
from src.evaluation import evaluate_model, cross_validate_model
from src.utils import save_model, load_model, plot_rmse_comparison

print('Importaciones correctas ✓')

## 2. Carga de datos

Descarga el dataset desde https://grouplens.org/datasets/movielens/100k/  
y coloca `ratings.csv` y `movies.csv` en la carpeta `data/`.

In [ ]:
ratings, movies = load_data()
print('\nPrimeras filas de ratings:')
display(ratings.head())
print('\nPrimeras filas de movies:')
display(movies.head())

## 3. Exploración de datos (EDA)

In [ ]:
print(f'Total de ratings  : {len(ratings):,}')
print(f'Usuarios únicos   : {ratings["userId"].nunique():,}')
print(f'Películas únicas  : {ratings["movieId"].nunique():,}')
print(f'Rango de ratings  : {ratings["rating"].min()} – {ratings["rating"].max()}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ratings['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribución de Ratings')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Frecuencia')

ratings_per_user = ratings.groupby('userId')['rating'].count()
axes[1].hist(ratings_per_user, bins=30, color='darkorange', edgecolor='white')
axes[1].set_title('Ratings por Usuario')
axes[1].set_xlabel('Número de ratings')
axes[1].set_ylabel('Usuarios')

plt.tight_layout()
plt.show()

## 4. Limpieza de datos

In [ ]:
ratings_clean, movies_clean = clean_data(
    ratings,
    movies,
    min_ratings_user=20,
    min_ratings_movie=10,
)
print(ratings_clean.dtypes)

## 5. Matriz usuario-película

In [ ]:
user_item_matrix = create_user_item_matrix(ratings_clean)
print(user_item_matrix.head())

## 6. Modelo KNN

In [ ]:
knn_model, matrix_imputed, imputer = train_knn_model(
    user_item_matrix,
    n_neighbors=20,
    metric='cosine',
)

In [ ]:
# Cambiar user_id por cualquier userId del dataset
USER_ID = ratings_clean['userId'].iloc[0]

knn_recs = get_knn_recommendations(
    user_id=USER_ID,
    user_item_matrix=user_item_matrix,
    knn_model=knn_model,
    matrix_imputed=matrix_imputed,
    movies_df=movies_clean,
    n_recommendations=10,
)
print(f'Top 10 recomendaciones KNN para el usuario {USER_ID}:')
display(knn_recs)

## 7. Modelo SVD

In [ ]:
svd_model, trainset, testset = train_svd_model(
    ratings_clean,
    n_factors=100,
    n_epochs=20,
)

In [ ]:
svd_recs = get_svd_recommendations(
    user_id=USER_ID,
    svd_model=svd_model,
    ratings_df=ratings_clean,
    movies_df=movies_clean,
    n_recommendations=10,
)
print(f'Top 10 recomendaciones SVD para el usuario {USER_ID}:')
display(svd_recs)

## 8. Evaluación de modelos (RMSE y MAE)

In [ ]:
svd_metrics = evaluate_model(svd_model, testset, model_name='SVD')
print(svd_metrics)

In [ ]:
from surprise import SVD as SurpriseSVD

cv_results = cross_validate_model(
    SurpriseSVD(n_factors=100, n_epochs=20),
    ratings_clean,
    n_splits=5,
)
print(cv_results)

## 9. Comparación KNN vs SVD

In [ ]:
# Sustituye estos valores por los obtenidos en tu ejecución
results = {
    'KNN': {'rmse': 0.9500, 'mae': 0.7400},
    'SVD': {'rmse': svd_metrics['rmse'], 'mae': svd_metrics['mae']},
}

plot_rmse_comparison(results)

## 10. Guardar modelos

In [ ]:
save_model(knn_model, 'knn_model.pkl')
save_model(svd_model, 'svd_model.pkl')

## 11. Conclusiones

| Modelo | RMSE  | MAE   |
|--------|-------|-------|
| KNN    | ~0.95 | ~0.74 |
| SVD    | ~0.87 | ~0.68 |

- **SVD** obtiene un RMSE más bajo, lo que indica predicciones más precisas.
- **KNN** es más interpretable y no requiere entrenamiento previo complejo.
- Ambos modelos son capaces de generar recomendaciones personalizadas útiles.